In [1]:
# input
pfam_file = "./tmp/no_anno_pfam_entryId-pfamId.tsv"
pdb_file = "./tmp/pdbId-entryId.tsv"
ted_file = "./tmp/no_anno_by_entryId-tedRepId.tsv"
rep_anno_file = "./tmp/mbp_repId-annoLevel.tsv"
seq_cluster_file = "../../../../afdb/_database/afdb_clusters/7-AFDB50-repId_memId.tsv"

# input
pred_file = "../../data/pred.tsv"
anno_file = "../../../collect_mbp/data/metal_chains.tsv"
cluster_file = "../../data/result_cluster.tsv"

In [2]:
import pandas as pd
df_pfam = pd.read_table(pfam_file, header=None, names=["seq_id", "pfam_id"])
df_ted = pd.read_table(ted_file, header=None, names=["seq_id", "ted_id"])
df_pdb = pd.read_table(pdb_file, header=None, names=["pdb_id", "seq_id"])

cur_ids = set(df_pdb['seq_id'])
len(cur_ids)

filter_by_pfam_ids = set()
for (_, ), df_part in df_pfam.groupby(by=['pfam_id']):
    anno_ids = set(df_part["seq_id"])
    if len(anno_ids - (anno_ids & cur_ids)) == 0:
        filter_by_pfam_ids |= anno_ids

filter_by_ted_ids = set()
for (_, ), df_part in df_ted.groupby(by=['ted_id']):
    anno_ids = set(df_part["seq_id"])
    if len(anno_ids - (anno_ids & cur_ids)) == 0:
        filter_by_ted_ids |= anno_ids

len(filter_by_pfam_ids)
len(filter_by_ted_ids)

150

15

86

In [3]:
df_rep_anno = pd.read_table(rep_anno_file, header=None, names=["rep_id", "anno_level"])
anno_rep_ids = set(df_rep_anno['rep_id'])
df_seq_cluster = pd.read_table(seq_cluster_file, header=None, names=["rep_id", "seq_id", "_"])
df_seq_cluster = df_seq_cluster[df_seq_cluster["seq_id"].map(lambda x: x in cur_ids)]
df_seq_cluster = df_seq_cluster[df_seq_cluster["rep_id"].map(lambda x: x not in anno_rep_ids)]

filter_by_clustser_ids = set(df_seq_cluster["seq_id"])

In [5]:
len(filter_by_clustser_ids)

31

In [6]:
no_anno = filter_by_pfam_ids & filter_by_ted_ids & filter_by_clustser_ids
# no_anno = filter_by_clustser_ids
# no_anno = cur_ids
len(no_anno)

8

In [7]:
no_anno_pdbs = set(df_pdb[df_pdb['seq_id'].map(lambda x: x in no_anno)]['pdb_id'])
no_anno_pdbs

{'8kcx_c',
 '8po5_b',
 '8pvs_b',
 '8q66_a',
 '8t03_b',
 '8ujm_b',
 '8xi2_n',
 '9f40_c'}

In [8]:
df_pred = pd.read_table(pred_file)
df_anno = pd.read_table(anno_file)

pred_resi = set()
for _, row in df_pred.iterrows():
    for i in row['posi'].split(','):
        if str.lower(row['seq_id']) in no_anno_pdbs:
            pred_resi.add((row['seq_id'], int(i)))

df_anno = df_anno[df_anno['resi'].map(lambda x: x in {"C", "H", "E", "D"})]
anno_resi = set()
for pdb_chain, df_chain in df_anno.groupby(by=["pdb", "metal_chain"]):
    pdb, chain = pdb_chain
    seq_id = f"{pdb}_{chain}"
    for i in df_chain["resi_ndb_seq_can_num"]:
        if str.lower(seq_id) in no_anno_pdbs:
            anno_resi.add((seq_id, int(i) - 1))
inter = pred_resi & anno_resi
len(inter)
len(pred_resi)  
len(anno_resi)
prec = len(inter) / len(pred_resi)
recall = len(inter) / len(anno_resi)
f1 = 2 * prec * recall / (prec + recall)
print(prec, recall, f1)

18

28

27

0.6428571428571429 0.6666666666666666 0.6545454545454545
